# Palmer Penguins Preprocessing

## Import libraries and load both datasets

In [66]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

DATA_DIR = Path.cwd() / 'data'
OUTPUT_DIR = DATA_DIR / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

penguins = pd.read_csv(DATA_DIR / 'penguins.csv')
penguins_raw = pd.read_csv(DATA_DIR / 'penguins_raw.csv')

print('penguins shape:', penguins.shape)
print('penguins_raw shape:', penguins_raw.shape)

penguins shape: (344, 7)
penguins_raw shape: (344, 17)


## Preview both datasets

In [67]:
display(penguins.head())
display(penguins_raw.head())

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.100,18.700,181.000,"3,750.000",MALE
1,Adelie,Torgersen,39.500,17.400,186.000,"3,800.000",FEMALE
2,Adelie,Torgersen,40.300,18.000,195.000,"3,250.000",FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.700,19.300,193.000,"3,450.000",FEMALE


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.100,18.700,181.000,"3,750.000",MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.500,17.400,186.000,"3,800.000",FEMALE,8.950,-24.695,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.300,18.000,195.000,"3,250.000",FEMALE,8.368,-25.333,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.700,19.300,193.000,"3,450.000",FEMALE,8.767,-25.324,NaN


## Check missing values in both datasets

In [68]:
penguins_missing = pd.DataFrame({
    'missing_count': penguins.isna().sum(),
    'missing_percent': (penguins.isna().mean() * 100).round(2)
})

penguins_raw_missing = pd.DataFrame({
    'missing_count': penguins_raw.isna().sum(),
    'missing_percent': (penguins_raw.isna().mean() * 100).round(2)
})

display(penguins_missing)
display(penguins_raw_missing)

,missing_count,missing_percent
species,0,0.000
island,0,0.000
bill_length_mm,2,0.580
bill_depth_mm,2,0.580
flipper_length_mm,2,0.580
body_mass_g,2,0.580
sex,11,3.200


,missing_count,missing_percent
studyName,0,0.000
Sample Number,0,0.000
Species,0,0.000
Region,0,0.000
Island,0,0.000
Stage,0,0.000
Individual ID,0,0.000
Clutch Completion,0,0.000
Date Egg,0,0.000
Culmen Length (mm),2,0.580


In [69]:
penguins_clean = penguins.copy()
penguins_raw_clean = penguins_raw.copy()

## Standardize important text columns

In [70]:
penguins_clean['species'] = penguins_clean['species'].astype('string').str.strip().str.title()
penguins_clean['island'] = penguins_clean['island'].astype('string').str.strip().str.title()
penguins_clean['sex'] = (
    penguins_clean['sex']
    .astype('string')
    .str.strip()
    .str.upper()
    .replace({'MALE': 'Male', 'FEMALE': 'Female'})
)

penguins_raw_clean['Species'] = penguins_raw_clean['Species'].astype('string').str.strip()
penguins_raw_clean['Island'] = penguins_raw_clean['Island'].astype('string').str.strip().str.title()
penguins_raw_clean['Sex'] = (
    penguins_raw_clean['Sex']
    .astype('string')
    .str.strip()
    .str.upper()
    .replace({'MALE': 'Male', 'FEMALE': 'Female', '.': pd.NA})
)

display(penguins_clean[['species', 'island', 'sex']].head())
display(penguins_raw_clean[['Species', 'Island', 'Sex']].head())

,species,island,sex
0,Adelie,Torgersen,Male
1,Adelie,Torgersen,Female
2,Adelie,Torgersen,Female
3,Adelie,Torgersen,<NA>
4,Adelie,Torgersen,Female


,Species,Island,Sex
0,Adelie Penguin (Pygoscelis adeliae),Torgersen,Male
1,Adelie Penguin (Pygoscelis adeliae),Torgersen,Female
2,Adelie Penguin (Pygoscelis adeliae),Torgersen,Female
3,Adelie Penguin (Pygoscelis adeliae),Torgersen,<NA>
4,Adelie Penguin (Pygoscelis adeliae),Torgersen,Female


## Handle missing values

fills missing numeric values with the median and missing sex values with `Unknown`.

In [71]:
penguins_numeric = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
penguins_raw_numeric = ['Culmen Length (mm)', 'Culmen Depth (mm)', 'Flipper Length (mm)', 'Body Mass (g)']

for col in penguins_numeric:
    penguins_clean[col] = penguins_clean[col].fillna(penguins_clean[col].median())

for col in penguins_raw_numeric:
    penguins_raw_clean[col] = penguins_raw_clean[col].fillna(penguins_raw_clean[col].median())

penguins_clean['sex'] = penguins_clean['sex'].fillna('Unknown')
penguins_raw_clean['Sex'] = penguins_raw_clean['Sex'].fillna('Unknown')

display(penguins_clean.isna().sum().to_frame('penguins_missing_after'))
display(penguins_raw_clean[['Culmen Length (mm)', 'Culmen Depth (mm)', 'Flipper Length (mm)', 'Body Mass (g)', 'Sex']].isna().sum().to_frame('penguins_raw_missing_after'))

,penguins_missing_after
species,0
island,0
bill_length_mm,0
bill_depth_mm,0
flipper_length_mm,0
body_mass_g,0
sex,0


,penguins_raw_missing_after
Culmen Length (mm),0
Culmen Depth (mm),0
Flipper Length (mm),0
Body Mass (g),0
Sex,0


## Detect outliers with the IQR rule

In [72]:
def iqr_flag(series: pd.Series) -> pd.Series:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

for col in penguins_numeric:
    penguins_clean[f'{col}_outlier'] = iqr_flag(penguins_clean[col])

for col in penguins_raw_numeric:
    safe_name = col.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_').replace('.', '').replace('-', '_')
    penguins_raw_clean[f'{safe_name}_outlier'] = iqr_flag(penguins_raw_clean[col])

penguins_clean['is_any_outlier'] = penguins_clean[[f'{col}_outlier' for col in penguins_numeric]].any(axis=1)

raw_outlier_cols = [col for col in penguins_raw_clean.columns if col.endswith('_outlier')]
penguins_raw_clean['is_any_outlier'] = penguins_raw_clean[raw_outlier_cols].any(axis=1)

display(penguins_clean[[f'{col}_outlier' for col in penguins_numeric] + ['is_any_outlier']].sum().to_frame('penguins_outlier_count'))
display(penguins_raw_clean[raw_outlier_cols + ['is_any_outlier']].sum().to_frame('penguins_raw_outlier_count'))

,penguins_outlier_count
bill_length_mm_outlier,0
bill_depth_mm_outlier,0
flipper_length_mm_outlier,0
body_mass_g_outlier,0
is_any_outlier,0


,penguins_raw_outlier_count
Culmen_Length_mm_outlier,0
Culmen_Depth_mm_outlier,0
Flipper_Length_mm_outlier,0
Body_Mass_g_outlier,0
is_any_outlier,0


## Create simple derived features

In [73]:
penguins_clean['body_mass_kg'] = penguins_clean['body_mass_g'] / 1000
penguins_clean['bill_area_mm2'] = penguins_clean['bill_length_mm'] * penguins_clean['bill_depth_mm']
penguins_clean['bill_ratio'] = penguins_clean['bill_length_mm'] / penguins_clean['bill_depth_mm']
penguins_clean['flipper_mass_ratio'] = penguins_clean['flipper_length_mm'] / penguins_clean['body_mass_kg']

penguins_raw_clean['body_mass_kg'] = penguins_raw_clean['Body Mass (g)'] / 1000
penguins_raw_clean['bill_area_mm2'] = penguins_raw_clean['Culmen Length (mm)'] * penguins_raw_clean['Culmen Depth (mm)']
penguins_raw_clean['bill_ratio'] = penguins_raw_clean['Culmen Length (mm)'] / penguins_raw_clean['Culmen Depth (mm)']
penguins_raw_clean['flipper_mass_ratio'] = penguins_raw_clean['Flipper Length (mm)'] / penguins_raw_clean['body_mass_kg']

display(penguins_clean[['body_mass_kg', 'bill_area_mm2', 'bill_ratio', 'flipper_mass_ratio']].head())
display(penguins_raw_clean[['body_mass_kg', 'bill_area_mm2', 'bill_ratio', 'flipper_mass_ratio']].head())

,body_mass_kg,bill_area_mm2,bill_ratio,flipper_mass_ratio
0,3.750,731.170,2.091,48.267
1,3.800,687.300,2.270,48.947
2,3.250,725.400,2.239,60.000
3,4.050,768.985,2.569,48.642
4,3.450,708.310,1.902,55.942


,body_mass_kg,bill_area_mm2,bill_ratio,flipper_mass_ratio
0,3.750,731.170,2.091,48.267
1,3.800,687.300,2.270,48.947
2,3.250,725.400,2.239,60.000
3,4.050,768.985,2.569,48.642
4,3.450,708.310,1.902,55.942


## Save the cleaned outputs


In [74]:
penguins_output = OUTPUT_DIR / 'penguins_cleaned.csv'
penguins_raw_output = OUTPUT_DIR / 'penguins_raw_cleaned.csv'

penguins_clean.to_csv(penguins_output, index=False)
penguins_raw_clean.to_csv(penguins_raw_output, index=False)

print('Saved:', penguins_output)
print('Saved:', penguins_raw_output)

Saved: c:\Users\MyPC\Documents\HW - Data Viz\Projec1\data\processed\penguins_cleaned.csv
Saved: c:\Users\MyPC\Documents\HW - Data Viz\Projec1\data\processed\penguins_raw_cleaned.csv
